In [0]:
%sql
USE flights_bronze;

### Załadowanie danych źródłowych

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.types import *

airlines_schema = StructType([
    StructField('iata_code', StringType(), True),
    StructField('airline', StringType(), True)
])

airlines_bronze = "flights_bronze.airlines"

df_airlines = spark.read.format("csv") \
    .schema(airlines_schema) \
    .option("header", "true") \
    .load("/Volumes/workspace/hurtownie_danych/cw7-projekt/airlines.csv")

df_airlines.write.mode("overwrite").saveAsTable(airlines_bronze)

In [0]:
%python
# show a few rows of df_airlines
display(df_airlines.limit(5))

iata_code,airline
UA,United Air Lines Inc.
AA,American Airlines Inc.
US,US Airways Inc.
F9,Frontier Airlines Inc.
B6,JetBlue Airways


### Proces ładowania przyrostowego

**Logika *MERGE* dla SCD1 - nadpisanie danych**  
Tabela *airlines* to tabela wymiaru pisująca atrybuty linii lotniczych. Jeśli cokolwiek, np. nazwa linii lub kod, się zmieni, wtedy nadpisaujemy stare dane. Nie musimy martwić się jednak o utratę danych historycznych przekazanych do warstwy silver, ponieważ tam następuje osobny prosces ładowania przyrostowego, odpowiedni dla SCD2.

In [0]:
%python

airlines_bronze = "flights_bronze.airlines"
new_file_path = "/Volumes/workspace/hurtownie_danych/cw7-projekt/airlines_new.csv"

# sprawdzamy czy nowy plik istnieje
file_exists = False
try:
  dbutils.fs.ls(new_file_path) # rzuci wyjątek, jeśli nie istnieje
  file_exists = True
except Exception as e:
  if "No such file or directory" in str(e):
    pass
  else:
    raise e

# jeśli istnieje - wczytujemy
if file_exists:
  df_airlines_new = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(new_file_path)

  # jeśli tabela nie istnieje, wykonujemy pierwszy pełen zapis, a jeśli istnieje, robimy MERGE INTO
  if not spark.catalog.tableExists(airlines_bronze):
    df_airlines_new.write.format("delta").saveAsTable(airlines_bronze)
  else:
    df_airlines_new.createOrReplaceTempView("src_airlines")

    spark.sql(f"""
      MERGE INTO {airlines_bronze} AS al
      USING src_airlines AS s
      ON al.iata_code = s.iata_code
      WHEN MATCHED THEN
        UPDATE SET 
        al.airline = s.airline
      WHEN NOT MATCHED THEN
        INSERT *
    """)
else:
  print(f"File {new_file_path} does not exist or have been already processed.")


File /Volumes/workspace/hurtownie_danych/cw7-projekt/airlines_new.csv does not exist or have been already processed.
